# Run the coupled MODFLOW + D-Flow FM + SWMM models

Scenario-driven runner for the Greenport coupled models. A *scenario* is one
combination of:

- **D-Flow FM discretization** (`resolution`: `coarse` / `medium` / `high`),
- **MODFLOW &harr; D-Flow FM coupling frequency** (`mf_couple_freq_hours`), and
- **number of SWMM &harr; MODFLOW connections** (`n_junctions`, capped to the
  junctions actually available).

For each scenario the notebook:
1. selects the D-Flow FM grid via `liss_settings.get_dflow_control_path`;
2. resolves the SWMM&ndash;MODFLOW connections and rebuilds the MODFLOW SWMM well
   package (`gwf_swmm.wel`);
3. copies the matching sewer tracer file
   `data/GP/Sewer_sourcesink_n{n_connections}__{tag}.bc` into the D-Flow FM run
   directory;
4. runs the three models coupled with a variable D-Flow-FM time step;
5. saves MODFLOW, SWMM, and D-Flow tracer results to a **scenario-named**
   directory for the step3 plotting notebooks; and
6. regenerates the sewer tracer `.bc` from this run's SWMM results.

> Derived from `step2_run_dflow-modflow_variable_dt_BNB.ipynb` and
> `step2_run_dflow-modflow_variable_dt.ipynb`.

## Imports

In [ ]:
import os
import time
import datetime
import shutil
import sys
import pathlib as pl

import numpy as np
import pandas as pd
import geopandas as gpd
import xugrid

import flopy
from modflowapi import ModflowApi
from modflowapi.extensions import ApiSimulation
from bmi.wrapper import BMIWrapper

import pyswmm
from pyswmm import Simulation, Nodes, Output

In [ ]:
sys.path.append("../common")
from liss_settings import (
    libmf6,
    get_dflow_control_path,
    get_dflow_grid_name,
    get_dflow_dtuser,
    get_modflow_grid_name,
    get_modflow_coupling_tag,
    silent,
    verbosity,
)
from swmm_mf_connect import intersect_points_grid

## Scenario configuration

**Edit these values to define the scenario.** Everything downstream (tags,
tracer file, output directory, well count, conductance scaling) is derived from
them &mdash; keyed on the *actual* number of connections resolved below.

In [ ]:
domain = "gp"                     # model domain
boundary_condition = "chd"        # MODFLOW coastal BC: "chd" or "ghb"
resolution = "coarse"             # D-Flow FM discretization: "coarse" | "medium" | "high"
mf_couple_freq_hours = 24.0       # MODFLOW <-> D-Flow FM coupling frequency (hours)
n_junctions = 500                 # requested # of SWMM <-> MODFLOW connections
                                  # (capped to the junctions actually available)

# SWMM input model (detailed Greenport sewer model)
swmm_inp_name = "greenport_detailedsewer_v4_nosub175.inp"

# The SWMM -> MODFLOW inflow coefficients in update_swmm were calibrated for this
# many connections; the per-connection coefficient is rescaled by
# (n_original / n_connections) so the TOTAL exchange conductance is constant.
n_original = 12

### Derived grid / coupling configuration

In [ ]:
control_path = get_dflow_control_path(domain, resolution)
dflow_grid_name = get_dflow_grid_name(control_path)
dflowfm_dtuser = get_dflow_dtuser(control_path)
mf_grid_name = get_modflow_grid_name(domain=domain, boundary_condition=boundary_condition)
mf_tag = get_modflow_coupling_tag(mf_couple_freq_hours)

mf_couple_freq = mf_couple_freq_hours * 60.0 * 60.0
dflow_per_mf = int(mf_couple_freq / dflowfm_dtuser)      # D-Flow steps per MODFLOW step
mf_couple_nstp = int(86400.0 / (dflow_per_mf * dflowfm_dtuser))

print(f"D-Flow grid   : {dflow_grid_name}  (DtUser={dflowfm_dtuser}s)")
print(f"MODFLOW grid  : {mf_grid_name}")
print(f"coupling tag  : {mf_tag}  ({mf_couple_freq_hours} h, {dflow_per_mf} D-Flow steps / MF step)")

### Unit conversions and coupling constants

In [ ]:
d2sec = 24.0 * 60.0 * 60.0
hrs2sec = 60.0 * 60.0
m2ft = 3.28081
cfd2cms = 1.0 / ((m2ft**3) * 86400.0)

HDRY = -1e30
DEPTH_MIN = 0.1

str(libmf6), libmf6.is_file()

## SWMM &harr; MODFLOW connections (resolved first)

`intersect_points_grid` samples `n_junctions` SWMM junctions, intersects them
with the MODFLOW grid, and returns the (layer, row, column) cell for each. It
**caps the request to the junctions actually available**, so the returned
`n_connections` is the actual number of connections &mdash; and that value drives
the scenario name, the tracer-bc lookup, the well package, and the conductance
scaling.

In [ ]:
(n_connections, junctions, mf6_cells, swmm_inverts, possible_junctions) = intersect_points_grid(
    domain=domain,
    boundary_condition=boundary_condition,
    n_junctions=n_junctions,
)
print(f"requested {n_junctions} -> {n_connections} actual SWMM <-> MODFLOW connections")
print(f"unique MODFLOW cells: {len(set(mf6_cells.values()))}")

# rescale the inflow coefficients so the TOTAL conductance is constant
conductance_scale = n_original / n_connections
print(f"conductance scale = n_original/n_connections = {n_original}/{n_connections} = {conductance_scale:.5f}")

### Scenario name and output location

In [ ]:
scenario = f"{domain}_{resolution}_{mf_tag}_n{n_connections:03d}"
results_ws = pl.Path(f"../results/{domain}/{scenario}").resolve()
results_ws.mkdir(parents=True, exist_ok=True)
print("scenario   :", scenario)
print("results ->  ", results_ws)

### Select the sewer tracer file for this scenario

The D-Flow FM `[SourceSink]` tracer forcing is a pre-tabulated SWMM outflow,
named by the actual number of connections and the coupling tag. It is copied
into the D-Flow FM run directory (as `Sewer_sourcesink.bc`, matching the
`FlowFM_bnd.ext` reference) before D-Flow FM is initialized.

In [ ]:
tracer_bc_src = pl.Path(f"../data/{domain.upper()}/Sewer_sourcesink_n{n_connections:03d}__{mf_tag}.bc")
print(tracer_bc_src, tracer_bc_src.is_file())
assert tracer_bc_src.is_file(), (
    f"no tracer bc for n{n_connections:03d} / {mf_tag}: {tracer_bc_src}\n"
    f"regenerate it with data/{domain.upper()}/update_files.py (n_junctions={n_connections}) first."
)

## D-Flow FM &rarr; MODFLOW mapping weights

Produced by `step1a` (GHB) and `step1b` (CHD), keyed on the D-Flow + MODFLOW grid
names.

In [ ]:
ghb_map = pl.Path(f"../mapping/{domain}/dflow_{dflow_grid_name}_to_{mf_grid_name}_ghb.npz")
chd_map = pl.Path(f"../mapping/{domain}/dflow_{dflow_grid_name}_to_{mf_grid_name}_chd.npz")
if not ghb_map.is_file():
    ghb_map = pl.Path(f"../mapping/dflow_{dflow_grid_name}_to_{mf_grid_name}_ghb.npz")
if not chd_map.is_file():
    chd_map = pl.Path(f"../mapping/dflow_{dflow_grid_name}_to_{mf_grid_name}_chd.npz")
print(ghb_map, ghb_map.is_file())
print(chd_map, chd_map.is_file())

npz = np.load(ghb_map)
dflow2mfghb, ghbmask, ghb2qext = npz["dflow2mfghb"], npz["ghbmask"], npz["ghb2qext"]
npz = np.load(chd_map)
dflow2mfchd, chdmask, chd2qext = npz["dflow2mfchd"], npz["chdmask"], npz["chd2qext"]
print("ghb:", dflow2mfghb.shape, "chd:", dflow2mfchd.shape)

## Load the base MODFLOW model into the scenario run directory

In [ ]:
mf_base_path = pl.Path(f"../modflow/{mf_grid_name}/base/").resolve()
mf_run_path = pl.Path(f"../modflow/{mf_grid_name}/run_{scenario}/").resolve()

sim = flopy.mf6.MFSimulation.load(sim_ws=mf_base_path, verbosity_level=verbosity())
gwf = sim.get_model()
sim.set_sim_path(mf_run_path)
(mf_run_path / "outputs").mkdir(parents=True, exist_ok=True)

### Set the MODFLOW time steps per stress period for the coupling frequency

In [ ]:
tdis = sim.get_package("TDIS")
perioddata = tdis.perioddata.array
perioddata["nstp"] = mf_couple_nstp
tdis.perioddata = perioddata

### Rebuild the SWMM well package from the connections

One WEL entry per SWMM connection (from `mf6_cells`), so the number of wells
matches the number of connections for this scenario. Multiple junctions may land
in one cell; MODFLOW sums their flux.

In [ ]:
swmm_well_spd = []
swmm_well_obs = []
for name, (lay, row, col) in mf6_cells.items():
    swmm_well_spd.append([lay, row, col, 0.0])          # Q filled in each step
    swmm_well_obs.append((str(name), "WEL", (lay, row, col)))

wel_swmm = flopy.mf6.ModflowGwfwel(
    gwf,
    save_flows=True,
    boundnames=True,
    stress_period_data={0: swmm_well_spd},
    observations={"outputs/swmm_well_obs.csv": swmm_well_obs},
    pname="SWMM",
)
sim.write_simulation(silent=silent())

### Base GHB / CHD stress-period data (templates used each step)

In [ ]:
ghb_data0 = gwf.ghb.stress_period_data.get_dataframe()[0]
assert ghb_data0.shape[0] == ghbmask.shape[0]

chd_surface = gwf.get_package("chd_surface")
chd_data0 = chd_surface.stress_period_data.get_dataframe()[0]
assert chd_data0.shape[0] == chdmask.shape[0]

## Set up and initialize D-Flow FM

In [ ]:
dflow_dirpath = (control_path.parent.parent.parent / "dflowfm_dll.2026.01").resolve()
dflow_base = control_path.parent
dflow_working = dflow_base.parent / "run"
dflow_config = dflow_working / "FlowFM.mdu"

# reset the working (run) copy from base
if dflow_working.is_dir():
    shutil.rmtree(dflow_working)
shutil.copytree(dflow_base, dflow_working)
(dflow_working / "output").mkdir(parents=True, exist_ok=True)

# copy the scenario tracer file in as the source/sink bc referenced by FlowFM_bnd.ext
shutil.copyfile(tracer_bc_src, dflow_working / "Sewer_sourcesink.bc")
print("D-Flow working dir:", dflow_working)

In [ ]:
# make the D-Flow FM dll discoverable, then initialize via BMI
os.environ["PATH"] = str(dflow_dirpath) + os.pathsep + os.environ["PATH"]
assert (dflow_dirpath / "dflowfm.dll").is_file(), dflow_dirpath

dflowfm = BMIWrapper(engine="dflowfm", configfile=str(dflow_config))
dflowfm.initialize()   # NOTE: this changes the working directory to the D-Flow run dir

### Grid variables and the source/sink exchange array

In [ ]:
ndxi = int(dflowfm.get_var("ndxi"))
ndx = int(dflowfm.get_var("ndx"))
qext = np.zeros(ndx)          # groundwater exchange pushed into D-Flow FM
qext_cum = np.zeros(ndx)
vextcum = dflowfm.get_var("vextcum")

def _read_mdu(path):
    cfg = {}
    for line in open(path):
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = [s.strip() for s in line.split("=", 1)]
        cfg[k.lower()] = v
    return cfg

_mdu = _read_mdu(dflow_config)
dflow_start_date = datetime.datetime.strptime(_mdu["refdate"].split()[0], "%Y%m%d")
dflow_end_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_end_time())
print("D-Flow:", dflow_start_date, "->", dflow_end_date, f"(ndxi={ndxi})")

## Initialize SWMM

In [ ]:
swmm_path = pl.Path(f"../swmm/{domain}/{swmm_inp_name}").resolve()
print(swmm_path, swmm_path.is_file())

for ext in (".out", ".rpt"):        # remove stale SWMM outputs
    p = swmm_path.with_suffix(ext)
    if p.is_file():
        p.unlink()

swmm_sim = pyswmm.Simulation(str(swmm_path))
with pyswmm.Simulation(str(swmm_path)) as _s:
    swmm_start_date, swmm_end_date = _s.start_time, _s.end_time
print("SWMM:", swmm_start_date, "->", swmm_end_date)

swmm_nodes = {j: pyswmm.Nodes(swmm_sim)[j] for j in junctions}
swmm_sim.start()

## Initialize MODFLOW via the MODFLOW API

In [ ]:
mf6 = ModflowApi(str(libmf6), working_directory=str(mf_run_path))
mf6.initialize()

apisim = ApiSimulation.load(mf6)
apiml = apisim.get_model()
sewer_flow = apiml.get_package("swmm")           # the SWMM well package handle
swmm_dtype = [("nodelist", "O"), ("q", float)]

### MODFLOW variable pointers (GHB bhead/cond, CHD head, boundary flows)

In [ ]:
ghb_bhead_ptr = mf6.get_value_ptr(mf6.get_var_address("BHEAD", "GWF", "GHB"))
ghb_cond_ptr = mf6.get_value_ptr(mf6.get_var_address("COND", "GWF", "GHB"))
ghb_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "GHB")

chd_head_ptr = mf6.get_value_ptr(mf6.get_var_address("HEAD", "GWF", "chd_surface"))
chd_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "chd_surface")

### Result dictionaries (keyed by coupling step)

In [ ]:
ghb_elev_dict, ghb_cond_dict, chd_elev_dict = {}, {}, {}
qext_dict, swmm_q_dict = {}, {}

## Coupling exchange functions

In [ ]:
def update_mf(key, s, d):
    """D-Flow FM water level (s) -> MODFLOW GHB bhead/cond and CHD head."""
    mask = d == 0.0
    s[mask] = 0.0
    mult = np.full(d.shape, 1.0)
    mult[mask] = 0.0

    ghb_head = ghb_data0["bhead"].to_numpy()
    ghb_head[ghbmask] = dflow2mfghb.dot(s)[ghbmask] * m2ft
    ghb_cond = ghb_data0["cond"].to_numpy()
    ghb_cond[ghbmask] = ghb_cond[ghbmask] * dflow2mfghb.dot(mult)[ghbmask]
    ghb_bhead_ptr[:] = ghb_head[:]
    ghb_cond_ptr[:] = ghb_cond[:]

    chd_head = chd_data0["head"].to_numpy()
    chd_head[chdmask] = dflow2mfchd.dot(s)[chdmask] * m2ft
    chd_head_ptr[:] = chd_head[:]

    ghb_elev_dict[key] = ghb_head.copy()
    ghb_cond_dict[key] = ghb_cond.copy()
    chd_elev_dict[key] = chd_head.copy()

In [ ]:
def update_dflow(key, d):
    """MODFLOW boundary flows -> D-Flow FM groundwater exchange (qext)."""
    ghb_flow = -mf6.get_value(ghb_flow_tag) * cfd2cms
    dflow_qext_ghb = ghb2qext.dot(ghb_flow)
    dflow_qext_ghb[d == 0.0] = 0.0

    chd_flow = -mf6.get_value(chd_flow_tag) * cfd2cms
    dflow_qext_chd = chd2qext.dot(chd_flow)
    dflow_qext_chd[d == 0.0] = 0.0

    dflow_qext = dflow_qext_ghb + dflow_qext_chd
    qext_cum[:ndxi] += dflow_qext[:ndxi]
    qext[:ndxi] = dflow_qext[:ndxi]
    dflowfm.set_var("qext", qext)

    qext_dict[key] = qext[:ndxi].copy()

In [ ]:
def update_swmm(key):
    """MODFLOW head vs SWMM invert -> SWMM inflow + MODFLOW SWMM well flux."""
    heads = apiml.X
    mf6_spd = []
    for name, node in swmm_nodes.items():
        lay, row, col = mf6_cells[name]
        head = heads[0, row, col]     # layer-0 head is adequate for the exchange
        pot = head - swmm_inverts[name] * m2ft
        # per-connection coefficient rescaled so total conductance is invariant
        if pot > 0.0:
            Q = pot * 0.001 * conductance_scale
        else:
            Q = pot * 0.0002 * conductance_scale
        node.generated_inflow(Q * cfd2cms)          # into SWMM (CMS)
        mf6_spd.append(((lay, row, col), -Q))       # out of MODFLOW

    sewer_flow.stress_period_data.values = np.array(mf6_spd, dtype=swmm_dtype)
    swmm_q_dict[key] = np.array(mf6_spd, dtype=swmm_dtype)["q"].copy()

## Run the coupled models

D-Flow FM advances every `DtUser`; every `dflow_per_mf` D-Flow steps the three
models exchange (one MODFLOW step + one SWMM step). This assumes the three models
share a start date; see the `_BNB` notebook for the date-staggered warm-up.

In [ ]:
idx, jdx = 0, 0
t0 = time.perf_counter()
current_time = dflowfm.get_current_time()
end_time = dflowfm.get_end_time()

while current_time <= end_time:
    idx += 1
    dflowfm.update()
    current_time = dflowfm.get_current_time()
    print(f"  {current_time/86400.:8.3f} d  {current_time/end_time:6.1%}  step {jdx:05d}", end="\r")

    if idx == int(dflow_per_mf):
        s = dflowfm.get_var("s1")[:ndxi]
        d = dflowfm.get_var("hs")[:ndxi]

        mf6.prepare_time_step(mf6.get_time_step())
        update_mf(str(jdx), s, d)
        mf6.do_time_step()
        mf6.finalize_time_step()
        update_dflow(str(jdx), d)

        update_swmm(str(jdx))
        swmm_sim.step_advance(int(mf6.get_time_step() * d2sec))
        try:
            swmm_sim.__next__()
        except StopIteration:
            break

        idx = 0
        jdx += 1

    if current_time == end_time:
        break

vextcum = dflowfm.get_var("vextcum")
print(f"\nrun time: {(time.perf_counter() - t0) / 60.0:.2f} min ({jdx} coupling steps)")

### Finalize the three models

In [ ]:
mf6.finalize()
swmm_sim.terminate_simulation()
swmm_sim.report()
swmm_sim.close()
dflowfm.finalize()

## Save MODFLOW results (scenario-named)

Exchange arrays as compressed `.npz` and the MODFLOW head/concentration output,
into the scenario results directory for the step3 plotting notebooks.

In [ ]:
np.savez_compressed(results_ws / "ghb_elev.npz", **ghb_elev_dict)
np.savez_compressed(results_ws / "ghb_cond.npz", **ghb_cond_dict)
np.savez_compressed(results_ws / "chd_elev.npz", **chd_elev_dict)
np.savez_compressed(results_ws / "qext.npz", **qext_dict)
np.savez_compressed(results_ws / "swmm_q.npz", **swmm_q_dict)

for out in (mf_run_path / "outputs").glob("*"):
    shutil.copy2(out, results_ws / out.name)
print("saved MODFLOW results ->", results_ws)

## Save SWMM results (scenario-named)

In [ ]:
for ext in (".out", ".rpt"):
    p = swmm_path.with_suffix(ext)
    if p.is_file():
        shutil.copy2(p, results_ws / f"swmm{ext}")
print("saved SWMM results ->", results_ws)

## Save the D-Flow FM tracer results (not the whole map file)

Extract only the sewage tracer field (`mesh2d_sewage`) plus the mesh geometry,
and write a compact scenario NetCDF &mdash; avoiding the multi-hundred-MB full
`FlowFM_map.nc`.

In [ ]:
map_path = dflow_working / "output" / "FlowFM_map.nc"
tracer_out = results_ws / "dflow_tracer.nc"
if map_path.is_file():
    ds = xugrid.open_dataset(map_path)
    keep = [v for v in ("mesh2d_sewage", "mesh2d_waterdepth") if v in ds]
    ds[keep].to_netcdf(tracer_out)     # UGRID geometry travels with the variables
    ds.close()
    print("saved D-Flow tracer ->", tracer_out)
else:
    print("no D-Flow map file found at", map_path)

## Regenerate the sewer tracer `.bc` from this run's SWMM results

Rebuild `Sewer_sourcesink_n{n_connections}__{tag}.bc` from the SWMM outflow so a
subsequent run uses updated sewer forcing. Reuses `write_bc` from
`data/GP/update_files.py` (no duplicated code).

> **TODO / verify:** confirm the outfall node id that drives the D-Flow FM
> source/sink, the tracer concentration, and the SWMM output units/reference
> time before relying on this.

In [ ]:
# reuse the .bc writer from data/GP/update_files.py
sys.path.append(f"../data/{domain.upper()}")
from update_files import write_bc, source_id, ref_time

outfall_node = "171"                 # TODO: confirm the D-Flow source/sink node
tracer_conc = 1000.0                 # kg/m3 sewage tracer (see update_files.py)

bc_out = pl.Path(f"../data/{domain.upper()}/Sewer_sourcesink_n{n_connections:03d}__{mf_tag}.bc")

with pyswmm.Output(str(swmm_path.with_suffix(".out"))) as out:
    series = out.node_series(outfall_node, pyswmm.output.NodeAttribute.TOTAL_INFLOW)
    ref = datetime.datetime.strptime(ref_time, "%Y-%m-%d %H:%M:%S")
    times = [(t - ref).total_seconds() for t in series.keys()]
    discharge = list(series.values())          # verify SWMM output units
    tracer = [tracer_conc] * len(times)

name = f"{source_id}_n{n_connections:03d}__{mf_tag}"
write_bc(bc_out, name, times, discharge, tracer, ref_time=ref_time)
print("regenerated tracer bc ->", bc_out)

---
### Notes / things to verify before a production run
- **Mapping weights** must exist for the chosen `resolution` (`step1a`/`step1b`).
- **Tracer bc** `Sewer_sourcesink_n{n_connections}__{tag}.bc` must exist for the
  actual connection count + coupling tag (regenerate via `update_files.py`).
- **SWMM inflow coefficients** in `update_swmm` (`0.001` / `0.0002`) are
  rescaled by `n_original / n_connections` (`n_original = 12`) so the total
  SWMM&harr;MODFLOW conductance is invariant to the number of connections.
- **bc regeneration** node id / units are scaffolded &mdash; confirm against
  `data/GP/update_files.py` and the SWMM model.
- The coupled run is long; run it outside this authoring session.